In [3]:
import difflib
import xml.etree.ElementTree as ET

In [8]:
def read_file(file_path):
    with open(file_path, 'r') as file:
        return file.readlines()

def show_diff(file1_lines, file2_lines):
    differ = difflib.Differ()
    diff = list(differ.compare(file1_lines, file2_lines))
    new_diff = []
    changed_line_numbers = []
    original_line_number = 0
    modified_line_number = 0
    lines_in_modified_to_mark_as_changed = []

    for i in range(len(diff)): # We gon assume for now that only diffs are changes
        line = diff[i]
        if line.startswith('- '):
            original_line_number += 1
        elif line.startswith('+ '):
            changed_line_numbers.append(modified_line_number + 1)
            modified_line_number += 1
        elif line.startswith('? '):
            continue
        else:
            original_line_number += 1
            modified_line_number += 1
        if line.startswith('- ') and (line[1:].strip()[1:5] == "step"): # Note changed!
            new_diff.append(line)
            new_diff.append(diff[i + 1])
            new_diff.append(diff[i + 2])
            lines_in_modified_to_mark_as_changed.append(modified_line_number + 1)
    return (''.join(diff), lines_in_modified_to_mark_as_changed)

def export_to_xml(xml_lines, lines_in_modified_to_mark_as_changed, output_file):
    for line_no in lines_in_modified_to_mark_as_changed:
        xml_lines.insert(line_no + 6, '   <notehead color="#31c854">normal</notehead>') # Make change appear as green note

    xml_string = '\n'.join(xml_lines)
    root = ET.fromstring(xml_string)
    tree = ET.ElementTree(root)
    with open(output_file, 'wb') as f:
        tree.write(f, encoding='utf-8', xml_declaration=True)

In [12]:
def main():
    old_file = 'Test_Score3'
    new_file = 'Test_Score4'
    file1_path = f'./sheet_music/{old_file}.musicxml'
    file2_path = f'./sheet_music/{new_file}.musicxml'
    output_file = f'./diffs/{old_file}_{new_file}_diff.musicxml'

    file1_lines = read_file(file1_path)
    file2_lines = read_file(file2_path)

    diff, lines_in_modified_to_mark_as_changed = show_diff(file1_lines, file2_lines)
    export_to_xml(file2_lines, lines_in_modified_to_mark_as_changed, output_file)
    
if __name__ == "__main__":
    main()